# Predicting Laptop Prices with Feature Engineering
## Practice Skeleton

**Short name (GitHub):** `LaptopPrice`

**Source lesson:** Predicting Laptop Prices (unit-bearing strings → numeric features → encode → Random Forest).

**Card:** `data/laptop_price.csv` — **823** listed laptops × **19** columns. Target `Price` is in INR (mean ≈ ₹76,745, median ₹64,990, max ₹441,990). Hardware fields arrive as text with units (`"8 GB"`, `"64-bit"`, `"4 stars"`, `"10th"`).

**How to use**
- Fill cells marked `# YOUR CODE HERE`. Keep the cheat-sheet and flowchart visible.
- Compare with `LaptopPrice_Solution.ipynb` only after an attempt.
- Data: `data/laptop_price.csv`. Charts: `laptopprice_*.png`.
- Clone the pipeline with `LaptopPrice_Reusable_Template.ipynb`.
- **Not a store-pricing engine and not financial advice.** Teaching catalog only.


## Inline cheat-sheet (keep this cell visible)

See also **`LaptopPrice_Cheatsheet.docx`**.

| Item | Code / rule |
|------|-------------|
| Unit strings | `df[col].astype(str).str.replace(' GB', '').astype(int)` — cast to `str` first so the cell is idempotent. |
| Composite feature | `total_storage = ssd + hdd`, then drop the parts if the model only needs capacity. |
| Sentinel missing | `"Not Available"` → `"0"` *before* stripping `"th"` on generation. |
| Cardinality rule | `< 5` uniques → `pd.get_dummies(..., drop_first=True)`. `≥ 5` → target (mean-Price) encoding. |
| Leakage | Fit target means on **train only**. Map test; unseen categories → `y_train.mean()`. |
| Split | `train_test_split(..., test_size=0.2, random_state=42)`. |
| Model | `RandomForestRegressor(random_state=42)` — default 100 trees. |
| Metrics | MAE in rupees + R². Always quote the **mean-predictor baseline**. |
| Importance | `feature_importances_` is split-count biased toward high-cardinality encodings. Confirm with `permutation_importance` on the test fold. |
| Never | Treat a leakage-inflated test R² as published skill. |


## Flowchart of the desired outcome

![LaptopPrice flow](laptopprice_flowchart.png)

Load the 823-row catalog → strip units from RAM / storage / GPU → clean OS-bit, stars, and generation sentinels → EDA (heatmap + RAM/GPU boxes) → encode by cardinality → 80/20 split → Random Forest vs linear / mean baselines → MAE + R² + top features → poke `n_estimators`, depth, noise, and sample size in the simulation cell.


## 0. Packages


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

sns.set_theme(style="whitegrid")


## 1. Load and inspect the catalog

823 rows. Many columns that *look* numeric (`ram_gb`, `ssd`, `os_bit`, `rating`) are stored as `object` because of units.

**Task**
- Load `data/laptop_price.csv` into `df`.
- Print `.head()`, `.info()`, `.shape`, and `.nunique()`.
- List object columns that should become numbers.


In [ ]:
# YOUR CODE HERE
df = None


## 2. Extract memory and storage features

Loop over `['ram_gb', 'ssd', 'hdd', 'graphic_card_gb']`, strip `" GB"`, cast to `int`.

Engineer `total_storage = ssd + hdd` and drop the two source columns.

Print `df[['ram_gb', 'total_storage', 'graphic_card_gb']].head()` to confirm.


In [ ]:
# YOUR CODE HERE


## 3. Clean OS bit, rating, and processor generation

Idempotent pattern: `.astype(str)` first.

- `os_bit`: strip `"-bit"` → int.
- `rating`: strip `" stars"` then `" star"` → int.
- `processor_gnrtn`: `"Not Available"` → `"0"`, then strip `"th"` → int.

Order matters on generation: replace the sentinel *before* stripping `"th"`.


In [ ]:
# YOUR CODE HERE


## 4. Exploratory data analysis

- `numeric_df = df.select_dtypes(include='number')`
- Correlation heatmap (`annot=True`, `cmap='coolwarm'`).
- Boxplot `ram_gb` vs `Price`.
- Boxplot `graphic_card_gb` vs `Price`.
- Optional: histogram of `Price` (expect right skew) and median price by `brand`.

Reference charts (after you plot, compare):

![heatmap](laptopprice_heatmap.png)
![RAM box](laptopprice_ram_box.png)
![GPU box](laptopprice_gpu_box.png)


In [ ]:
# YOUR CODE HERE
numeric_df = None


## 5. Encode categoricals and split

Identify object columns. For each:

- `nunique() < 5` → one-hot with `drop_first=True`, concat, drop original.
- else → replace the category with its mean `Price` (lesson version uses the *full* frame — note the leakage).

Then `X = df.drop(columns=['Price'])`, `y = df['Price']`, 80/20 split, `random_state=42`.

**Stretch (recommended):** split *first*, fit target means on train only. That version lives in the Alternate code section.


In [ ]:
# YOUR CODE HERE
X_train = X_test = y_train = y_test = None


## 6. Fit the Random Forest

`RandomForestRegressor(random_state=42)`, `.fit` on train, `.predict` on test → `y_pred`.


In [ ]:
# YOUR CODE HERE
rf_model = None
y_pred = None


## 7. Evaluate and rank features

Print MAE and R². Also print the **mean-predictor baseline** MAE so the forest has something to beat.

Top-5 impurity importances with `np.argsort` and a horizontal bar plot.

![importance](laptopprice_importance.png)
![pred vs actual](laptopprice_pred_actual.png)


In [ ]:
# YOUR CODE HERE
mae = r2 = None


## 8. Alternate code — same destination, different route

Work these after the main path. They should land near the same MAE / R² band.


### 8a. Regex extract instead of `str.replace`


In [ ]:
# YOUR CODE HERE
# Example idea: df['ram_gb'].astype(str).str.extract(r'(\d+)').astype(int)


### 8b. Leakage-safe target encoding (train means only)


In [ ]:
# YOUR CODE HERE
# Split raw X,y first. Map high-cardinality cols with train groupby means.
# Fill unseen test categories with y_train.mean().


### 8c. Linear / Ridge baseline on the same encoded matrix


In [ ]:
# YOUR CODE HERE
# LinearRegression() and Ridge(alpha=1.0). Compare MAE and R² to the forest.


### 8d. Permutation importance on the test fold


In [ ]:
# YOUR CODE HERE
# permutation_importance(rf_model, X_test, y_test, n_repeats=8, random_state=42)


## 9. More practice

1. Predict `np.log1p(Price)` then `np.expm1` the predictions. Does test MAE drop?
2. Keep `ssd` and `hdd` *instead of* `total_storage`. Does the forest prefer the split?
3. Drop `Number of Ratings` and `Number of Reviews` (popularity is not a spec). How much R² do you lose?
4. Fit only the `os == 'Windows'` slice. Does Apple's 28-row cluster distort the global fit?
5. Threshold a "premium" flag at Price ≥ 100000 and report how often the forest misses those tails.


In [ ]:
# YOUR CODE HERE — pick at least two practice items


## 10. Simulation — move three knobs, watch MAE / R²

Reference run: ![simulation](laptopprice_simulation.png)

Edit the parameters cell and re-run. Suggested starting point: `N_EST=100`, `MAX_DEPTH=None`, `NOISE_SD=0`, `SUBSAMPLE=1.0`.


In [ ]:
# knobs — edit these
N_EST = 100
MAX_DEPTH = None          # try 4, 8, 16, None
NOISE_SD = 0              # rupee noise added to y_train (try 5000, 15000)
SUBSAMPLE = 1.0           # fraction of training rows (try 0.4, 0.7)
RANDOM_STATE = 42


In [ ]:
# YOUR CODE HERE
# 1. Optionally subsample X_train / y_train.
# 2. Add N(0, NOISE_SD) to the training target.
# 3. Fit RandomForestRegressor(n_estimators=N_EST, max_depth=MAX_DEPTH, random_state=RANDOM_STATE).
# 4. Print test MAE and R² next to the default run (~₹12,401 / 0.725).


## 11. What this model can and cannot do

**Can**
- Rank which listed specs move price on *this* catalog (processor family, total storage, dedicated GPU).
- Beat a mean/median list-price guess by a wide margin (MAE ~₹12.4k vs ~₹31k).
- Stress-test encoding choices and show how leakage flatters a score.

**Cannot**
- Set a shelf price for a laptop that is not in the 823-row mix (no RAM=32 GB rows, almost no 64 GB).
- Separate brand prestige from the specs that co-occur with that brand (target encoding collapses them).
- Use review *counts* as a causal feature at listing time — those arrive after the product is sold.
- Generalize to US-dollar street prices, refurbished units, or 2026 configs without a refresh.

**Top applications of the same pipeline:** used cars, smartphones, apartments, cameras/lenses, hotel ADR — any listing table where capacity lives inside unit-bearing strings.
